# Agentic AI System — Warehouse Intelligence Assistant (WIA)

**Author:** Sébastien Bodrero
**Programme:** Woolf University / Udacity MSc in Artificial Intelligence
**Module:** Agentic AI Systems (Module 6)
**Date:** April 2026

---

This notebook implements a single-agent system called the **Warehouse Intelligence Assistant (WIA)** — an LLM-powered agent that helps warehouse operations managers make real-time fleet coordination decisions. The agent uses the Claude API for reasoning, calls simulated warehouse tools, maintains a decision log in memory, and applies explicit safeguards before issuing any routing recommendation.

**Notebook structure:**
1. [Task 1 — Agentic Task and System Scope](#task1)
2. [Task 2 — Agent Architecture](#task2)
3. [Task 3 — Implementation](#task3)
4. [Task 4 — Execution and Observed Behavior](#task4)
5. [Task 5 — Summary](#task5)
6. [Task 6 — Report Reference](#task6)
7. [Task 7 — Requirements](#task7)

---
## Setup — Imports and Configuration

In [1]:
import os
import json
import datetime
from collections import deque
from typing import Any

import anthropic

print(f"anthropic SDK version : {anthropic.__version__}")
print(f"Notebook executed at  : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

anthropic SDK version : 0.90.0
Notebook executed at  : 2026-04-19 18:14:13


<a id='task1'></a>
---
## Task 1 — Agentic Task and System Scope

### What the agent does

The **Warehouse Intelligence Assistant (WIA)** is a conversational agent that takes natural-language requests from a warehouse operations manager and translates them into concrete, safe fleet-coordination actions. A typical request might be:

> *"Route AGV-3 to pick zone C — what's the current inventory there and is the path clear?"*

The agent gathers context via tools (inventory levels, vehicle status, safe pathfinding), reasons about the best course of action, and produces a recommendation or escalates to a human operator when appropriate.

### Why an agentic approach is appropriate

A static classifier or rule engine cannot handle the **open-ended, multi-step nature** of warehouse coordination queries. The agent must:
- Decide *which* tools to call and *in what order* based on the query
- Integrate information across multiple tool results before reasoning
- Adapt its strategy when a tool returns unexpected results (e.g., zone blocked)
- Apply safety constraints that depend on runtime state, not static rules

This requires an LLM-driven reasoning loop — the hallmark of agentic design.

### Scope and boundaries

| In scope | Out of scope |
|---|---|
| Routing recommendations for named AGVs | Physically issuing commands to real hardware |
| Inventory level checks per zone | Persistent inventory database updates |
| Safe-path computation (BFS on a grid) | Full warehouse digital twin / simulation |
| Human escalation when safety zones are threatened | Multi-agent coordination between WIA instances |
| Decision logging for audit trail | Long-term persistent memory across sessions |

### Design type

**Single-agent** — one WIA instance handles one manager conversation. The agent is not persistent across sessions; each notebook execution is a fresh session. This scope is appropriate for the academic context and mirrors real-world "shift assistant" deployments where state is reset at shift change.

In [2]:
# ── Configuration constants ────────────────────────────────────────────────

# Zones where human operators are present; routing through these requires
# explicit escalation before the agent may recommend a path.
HUMAN_SAFETY_ZONES = {"H1", "H2", "H3", "MAIN_AISLE"}

# Maximum number of tool-call iterations per user turn (prevents runaway loops)
MAX_TOOL_ITERATIONS = 6

# Number of past decisions kept in the rolling decision log
MEMORY_WINDOW = 10

# Claude model to use for the agent's LLM backbone
MODEL_ID = "claude-haiku-4-5-20251001"   # fast & cost-efficient for this demo

print("Configuration loaded.")
print(f"  Safety zones   : {HUMAN_SAFETY_ZONES}")
print(f"  Max iterations : {MAX_TOOL_ITERATIONS}")
print(f"  Memory window  : {MEMORY_WINDOW}")
print(f"  LLM model      : {MODEL_ID}")


Configuration loaded.
  Safety zones   : {'MAIN_AISLE', 'H1', 'H3', 'H2'}
  Max iterations : 6
  Memory window  : 10
  LLM model      : claude-haiku-4-5-20251001


<a id='task2'></a>
---
## Task 2 — Agent Architecture

### Component overview

```
┌─────────────────────────────────────────────────────────────────┐
│                  Warehouse Intelligence Assistant                │
│                                                                 │
│  ┌───────────────┐    ┌──────────────────────────────────────┐  │
│  │  AgentMemory  │◄───│            WarehouseAgent            │  │
│  │               │    │                                      │  │
│  │ • conversation│    │  • System prompt (persona)           │  │
│  │   history     │    │  • run(user_query) → reasoning loop  │  │
│  │ • decision    │    │  • _dispatch_tool(name, args)        │  │
│  │   log         │    │  • _safety_check(tool, args)         │  │
│  └───────────────┘    └──────────────┬───────────────────────┘  │
│                                      │  tool_use / tool_result  │
│                       ┌──────────────▼───────────────────────┐  │
│                       │          WarehouseTools               │  │
│                       │                                      │  │
│                       │  check_inventory(zone_id)            │  │
│                       │  get_agent_status(agent_id)          │  │
│                       │  find_safe_path(start, end, blocked) │  │
│                       │  escalate_to_human(reason, urgency)  │  │
│                       └──────────────────────────────────────┘  │
│                                      │  Claude API              │
│                       ┌──────────────▼───────────────────────┐  │
│                       │         Anthropic Claude API          │  │
│                       │    (claude-haiku-4-5-20251001)        │  │
│                       └──────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### Reasoning loop (ReAct-style)

```
User query
    │
    ▼
[1] Build messages (system prompt + memory + query)
    │
    ▼
[2] Call Claude API with tool definitions
    │
    ├── stop_reason = "end_turn"  → return final text answer
    │
    └── stop_reason = "tool_use"
            │
            ▼
        [3] Safety check (is this tool call safe?)
                │
                ├── BLOCKED → auto-escalate, append warning, continue
                │
                └── OK → dispatch tool, get result
                            │
                            ▼
                        [4] Append tool_result to messages
                            │
                            ▼
                        [5] goto [2]  (until end_turn or MAX_TOOL_ITERATIONS)
```

### Design choices and tradeoffs

| Choice | Rationale | Tradeoff |
|---|---|---|
| Single-agent (not multi-agent) | Simpler credit assignment; sufficient for this scope | Cannot parallelise sub-tasks across specialised agents |
| Simulated tools (not real APIs) | No external infrastructure dependency; fully reproducible | Does not reflect real warehouse API latency or failures |
| In-session memory only | No persistent state complicates reproducibility and safety | Agent forgets decisions between notebook restarts |
| Claude Haiku (not Sonnet/Opus) | Faster, cheaper for iterative testing | Slightly less nuanced reasoning on ambiguous queries |
| Safety check before dispatch | Fail-safe: block unsafe paths *before* tool execution | Adds latency; may over-block edge cases |
| BFS pathfinding (not A*) | Simple, predictable, no heuristic tuning | Suboptimal for large grids; A* would be better in production |

<a id='task3'></a>
---
## Task 3 — Implementation

In [3]:
# ── Simulated warehouse state ──────────────────────────────────────────────
# In production this would come from a live WMS (Warehouse Management System).
# Here we use static Python dicts for full reproducibility.

WAREHOUSE_INVENTORY = {
    "A": {"item": "SKU-1001 (Electronics)",   "units": 142, "status": "normal"},
    "B": {"item": "SKU-2034 (Textiles)",       "units": 0,   "status": "empty"},
    "C": {"item": "SKU-3087 (Automotive)",     "units": 58,  "status": "low"},
    "D": {"item": "SKU-4112 (Pharmaceuticals)","units": 310, "status": "normal"},
    "E": {"item": "SKU-5200 (Food & Beverage)","units": 75,  "status": "normal"},
}

FLEET_STATUS = {
    "AGV-1": {"type": "forklift",     "location": "A",        "battery": 87,  "status": "idle"},
    "AGV-2": {"type": "conveyor-bot", "location": "CHARGING", "battery": 12,  "status": "charging"},
    "AGV-3": {"type": "forklift",     "location": "D",        "battery": 65,  "status": "idle"},
    "AGV-4": {"type": "drone",        "location": "H2",       "battery": 44,  "status": "busy"},
    "AGV-5": {"type": "conveyor-bot", "location": "E",        "battery": 91,  "status": "idle"},
}

# Simple 5×5 grid for BFS path-finding.  Zones are nodes; adjacency is direct.
ZONE_GRAPH = {
    "A":        ["B", "MAIN_AISLE"],
    "B":        ["A", "C", "H1"],
    "C":        ["B", "D", "MAIN_AISLE"],
    "D":        ["C", "E", "H2"],
    "E":        ["D", "MAIN_AISLE"],
    "H1":       ["B", "H2"],
    "H2":       ["H1", "H3", "D"],
    "H3":       ["H2", "MAIN_AISLE"],
    "MAIN_AISLE":["A", "C", "E", "H3", "CHARGING"],
    "CHARGING": ["MAIN_AISLE"],
}

print("Warehouse state loaded.")

Warehouse state loaded.


In [4]:
class WarehouseTools:
    """
    Simulated warehouse tool layer.

    Each method corresponds to one tool the agent may invoke.  All methods
    return a dict so results can be serialised directly to JSON for the
    Claude tool_result message.
    """

    def check_inventory(self, zone_id: str) -> dict:
        """
        Return current inventory levels for a warehouse zone.

        Parameters
        ----------
        zone_id : str
            Zone identifier (e.g. 'A', 'B', 'C', 'D', 'E').

        Returns
        -------
        dict
            Keys: zone, item, units, status.  'error' key present on failure.
        """
        zone_id = zone_id.upper()
        if zone_id not in WAREHOUSE_INVENTORY:
            return {"error": f"Zone '{zone_id}' not found in inventory system."}
        data = WAREHOUSE_INVENTORY[zone_id].copy()
        data["zone"] = zone_id
        return data

    def get_agent_status(self, agent_id: str) -> dict:
        """
        Return current status of a fleet vehicle (AGV, drone, conveyor-bot).

        Parameters
        ----------
        agent_id : str
            Vehicle identifier (e.g. 'AGV-1', 'AGV-3').

        Returns
        -------
        dict
            Keys: agent_id, type, location, battery, status.  'error' if unknown.
        """
        if agent_id not in FLEET_STATUS:
            return {"error": f"Vehicle '{agent_id}' not registered in fleet."}
        data = FLEET_STATUS[agent_id].copy()
        data["agent_id"] = agent_id
        return data

    def find_safe_path(self, start: str, end: str, blocked_zones: list[str]) -> dict:
        """
        Find a path from start to end in the warehouse zone graph using BFS,
        avoiding any explicitly blocked zones.

        Parameters
        ----------
        start : str
            Starting zone identifier.
        end : str
            Destination zone identifier.
        blocked_zones : list[str]
            Zones to exclude from the path (e.g. zones under maintenance).

        Returns
        -------
        dict
            Keys: path (list of zones), hops (int).  'error' if no path found.

        Notes
        -----
        BFS guarantees the shortest path in terms of number of zone transitions.
        It does not account for travel time or congestion.
        """
        start = start.upper()
        end   = end.upper()
        blocked = {z.upper() for z in (blocked_zones or [])}

        if start not in ZONE_GRAPH:
            return {"error": f"Unknown start zone '{start}'."}
        if end not in ZONE_GRAPH:
            return {"error": f"Unknown destination zone '{end}'."}

        # BFS
        queue   = deque([[start]])
        visited = {start}
        while queue:
            path = queue.popleft()
            node = path[-1]
            if node == end:
                return {"path": path, "hops": len(path) - 1}
            for neighbour in ZONE_GRAPH.get(node, []):
                if neighbour not in visited and neighbour not in blocked:
                    visited.add(neighbour)
                    queue.append(path + [neighbour])

        return {"error": f"No path from '{start}' to '{end}' avoiding {blocked}."}

    def escalate_to_human(self, reason: str, urgency: str) -> dict:
        """
        Log an escalation event and notify the human operator.

        In a production system this would send a push notification or page the
        shift supervisor.  Here we log the event and return a confirmation.

        Parameters
        ----------
        reason : str
            Human-readable description of why escalation is needed.
        urgency : str
            One of: 'low', 'medium', 'high', 'critical'.

        Returns
        -------
        dict
            Escalation record with timestamp and ticket_id.
        """
        ticket_id = f"ESC-{datetime.datetime.now().strftime('%H%M%S')}"
        record = {
            "ticket_id"  : ticket_id,
            "timestamp"  : datetime.datetime.now().isoformat(),
            "reason"     : reason,
            "urgency"    : urgency,
            "status"     : "OPEN",
            "message"    : (
                f"[{urgency.upper()}] Human operator notified — ticket {ticket_id}. "
                "Awaiting operator acknowledgement before proceeding."
            ),
        }
        print(f"  ⚠  ESCALATION RAISED: {ticket_id} [{urgency.upper()}] — {reason}")
        return record


tools_instance = WarehouseTools()
print("WarehouseTools initialised.")

WarehouseTools initialised.


In [5]:
class AgentMemory:
    """
    Lightweight in-session memory for the WIA agent.

    Maintains two structures:
    - conversation_history : list of Claude API message dicts for the current turn.
    - decision_log         : rolling deque of past decisions (capped at MEMORY_WINDOW).

    The decision log is injected into the system prompt at each turn so the
    agent can refer to recent actions without bloating the full message history.
    """

    def __init__(self, window: int = MEMORY_WINDOW):
        self.conversation_history: list[dict] = []
        self.decision_log: deque[dict] = deque(maxlen=window)

    def add_user_message(self, text: str) -> None:
        """Append a user message to the conversation history."""
        self.conversation_history.append({"role": "user", "content": text})

    def add_assistant_message(self, content: Any) -> None:
        """Append an assistant message (text or tool_use blocks) to history."""
        self.conversation_history.append({"role": "assistant", "content": content})

    def add_tool_result(self, tool_use_id: str, result: dict) -> None:
        """Append a tool_result message to the conversation history."""
        self.conversation_history.append({
            "role": "user",
            "content": [{
                "type"      : "tool_result",
                "tool_use_id": tool_use_id,
                "content"   : json.dumps(result),
            }],
        })

    def log_decision(self, query: str, action: str, outcome: str) -> None:
        """Record a completed decision in the rolling log."""
        self.decision_log.append({
            "timestamp": datetime.datetime.now().isoformat(),
            "query"    : query[:80],   # truncate for brevity
            "action"   : action,
            "outcome"  : outcome,
        })

    def get_decision_summary(self) -> str:
        """Return a compact text summary of recent decisions for the system prompt."""
        if not self.decision_log:
            return "No prior decisions in this session."
        lines = []
        for i, d in enumerate(self.decision_log, 1):
            lines.append(f"  {i}. [{d['timestamp'][:19]}] {d['query']} → {d['outcome']}")
        return "\n".join(lines)

    def reset_turn(self) -> None:
        """Clear conversation history while preserving the decision log."""
        self.conversation_history = []


print("AgentMemory class defined.")

AgentMemory class defined.


In [6]:
# ── Claude API tool schema definitions ────────────────────────────────────
# These are passed to the API so the model knows which tools it can call.

TOOL_SCHEMAS = [
    {
        "name": "check_inventory",
        "description": (
            "Look up current inventory levels for a specific warehouse zone. "
            "Returns item name, unit count, and stock status (normal / low / empty)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "zone_id": {
                    "type": "string",
                    "description": "Zone identifier, e.g. 'A', 'B', 'C', 'D', or 'E'.",
                }
            },
            "required": ["zone_id"],
        },
    },
    {
        "name": "get_agent_status",
        "description": (
            "Retrieve the current operational status of a fleet vehicle. "
            "Returns vehicle type, current location, battery percentage, and availability status."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "agent_id": {
                    "type": "string",
                    "description": "Vehicle identifier, e.g. 'AGV-1', 'AGV-3'.",
                }
            },
            "required": ["agent_id"],
        },
    },
    {
        "name": "find_safe_path",
        "description": (
            "Compute the shortest path between two warehouse zones while avoiding "
            "specified blocked zones. Uses BFS over the zone adjacency graph."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "start": {
                    "type": "string",
                    "description": "Starting zone identifier.",
                },
                "end": {
                    "type": "string",
                    "description": "Destination zone identifier.",
                },
                "blocked_zones": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of zone identifiers to avoid (e.g. under maintenance).",
                },
            },
            "required": ["start", "end"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": (
            "Raise a human escalation event. Use this when a routing request "
            "involves a human safety zone, when confidence is low, or when an "
            "irreversible action requires supervisor approval."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "reason": {
                    "type": "string",
                    "description": "Human-readable explanation of why escalation is needed.",
                },
                "urgency": {
                    "type": "string",
                    "enum": ["low", "medium", "high", "critical"],
                    "description": "Urgency level of the escalation.",
                },
            },
            "required": ["reason", "urgency"],
        },
    },
]

print(f"{len(TOOL_SCHEMAS)} tool schemas defined.")

4 tool schemas defined.


In [7]:
class WarehouseAgent:
    """
    Warehouse Intelligence Assistant — single-agent reasoning loop.

    The agent uses the Anthropic Claude API in a ReAct-style loop:
    think → tool call → observe → think → … → final answer.

    Safeguards
    ----------
    1. Iteration cap  : aborts if the model calls tools more than MAX_TOOL_ITERATIONS
                        times in one turn, preventing runaway loops.
    2. Safety zones   : any find_safe_path call whose computed path passes through
                        a HUMAN_SAFETY_ZONE triggers an auto-escalation before the
                        recommendation is returned.
    3. Decision log   : every completed turn is recorded in AgentMemory for audit.
    """

    SYSTEM_PROMPT_TEMPLATE = """You are the Warehouse Intelligence Assistant (WIA), \
an expert logistics AI embedded in a large distribution centre.

Your role is to help operations managers make safe, efficient fleet-coordination decisions.
You have access to four tools: check_inventory, get_agent_status, find_safe_path, and
escalate_to_human.  Always gather the facts you need before making a recommendation.

Safety rules you MUST follow:
- If a path passes through a human safety zone (H1, H2, H3, MAIN_AISLE), you MUST call
  escalate_to_human before recommending the route.
- If a vehicle's battery is below 20%, recommend charging instead of routing.
- If a vehicle status is 'busy' or 'charging', do not reroute it without noting the conflict.
- If you are uncertain about any critical aspect, call escalate_to_human.

Recent decision log (last {window} decisions):
{decision_log}
"""

    def __init__(self):
        self.client  = anthropic.Anthropic()   # uses ANTHROPIC_API_KEY env var
        self.tools   = tools_instance
        self.memory  = AgentMemory()

    def _build_system_prompt(self) -> str:
        """Inject the current decision log into the system prompt."""
        return self.SYSTEM_PROMPT_TEMPLATE.format(
            window       = MEMORY_WINDOW,
            decision_log = self.memory.get_decision_summary(),
        )

    def _safety_check(self, tool_name: str, tool_input: dict) -> str | None:
        """
        Pre-dispatch safety check.  Returns a warning string if the call
        violates a safety rule, or None if the call is safe to execute.

        Checks applied:
        - find_safe_path: if the proposed path traverses a human safety zone
          AND no blocked_zones override was provided, return a warning so the
          agent can decide to escalate.
        """
        if tool_name == "find_safe_path":
            start  = tool_input.get("start", "").upper()
            end    = tool_input.get("end", "").upper()
            # Run BFS without any extra blocks to discover the natural path
            natural = self.tools.find_safe_path(start, end, [])
            if "path" in natural:
                overlap = HUMAN_SAFETY_ZONES & set(natural["path"])
                if overlap:
                    return (
                        f"SAFETY WARNING: Natural shortest path from {start} to {end} "
                        f"passes through human safety zone(s): {overlap}. "
                        "Escalation required before issuing routing recommendation."
                    )
        return None

    def _dispatch_tool(self, tool_name: str, tool_input: dict) -> dict:
        """
        Route a tool_use request to the appropriate WarehouseTools method.

        Parameters
        ----------
        tool_name  : str  — Name of the tool (must match TOOL_SCHEMAS names)
        tool_input : dict — Arguments as parsed from the Claude API response

        Returns
        -------
        dict  — Tool result to be serialised into a tool_result message
        """
        dispatch_map = {
            "check_inventory"  : self.tools.check_inventory,
            "get_agent_status" : self.tools.get_agent_status,
            "find_safe_path"   : lambda args: self.tools.find_safe_path(
                                     args["start"], args["end"],
                                     args.get("blocked_zones", [])
                                 ),
            "escalate_to_human": lambda args: self.tools.escalate_to_human(
                                     args["reason"], args["urgency"]
                                 ),
        }
        handler = dispatch_map.get(tool_name)
        if handler is None:
            return {"error": f"Unknown tool '{tool_name}'."}
        try:
            if tool_name in ("check_inventory", "get_agent_status"):
                # These tools take a single positional string arg
                first_val = next(iter(tool_input.values()))
                return handler(first_val)
            return handler(tool_input)
        except Exception as exc:  # noqa: BLE001
            return {"error": str(exc)}

    def run(self, user_query: str, verbose: bool = True) -> str:
        """
        Process a user query through the full ReAct reasoning loop.

        Parameters
        ----------
        user_query : str   — Natural-language request from the warehouse manager
        verbose    : bool  — If True, print each reasoning step to stdout

        Returns
        -------
        str — The agent's final text response
        """
        self.memory.reset_turn()
        self.memory.add_user_message(user_query)

        if verbose:
            print(f"\n{'='*70}")
            print(f"USER: {user_query}")
            print(f"{'='*70}")

        iteration   = 0
        final_text  = ""

        while iteration < MAX_TOOL_ITERATIONS:
            response = self.client.messages.create(
                model      = MODEL_ID,
                max_tokens = 1024,
                system     = self._build_system_prompt(),
                tools      = TOOL_SCHEMAS,
                messages   = self.memory.conversation_history,
            )

            self.memory.add_assistant_message(response.content)

            if response.stop_reason == "end_turn":
                # Extract the final text block
                for block in response.content:
                    if hasattr(block, "text"):
                        final_text = block.text
                        break
                if verbose:
                    print(f"\nAGENT: {final_text}")
                break

            if response.stop_reason == "tool_use":
                for block in response.content:
                    if block.type != "tool_use":
                        continue

                    tool_name  = block.name
                    tool_input = block.input

                    if verbose:
                        print(f"\n  [iter {iteration+1}] TOOL CALL → {tool_name}({tool_input})")

                    # ── Safeguard: pre-dispatch safety check ──────────────
                    warning = self._safety_check(tool_name, tool_input)
                    if warning:
                        if verbose:
                            print(f"  !! SAFETY CHECK: {warning}")
                        # Inject warning as tool result so LLM can react
                        self.memory.add_tool_result(
                            block.id, {"safety_warning": warning}
                        )
                    else:
                        result = self._dispatch_tool(tool_name, tool_input)
                        if verbose:
                            print(f"  [result] {result}")
                        self.memory.add_tool_result(block.id, result)

                iteration += 1
            else:
                # Unexpected stop reason
                final_text = f"[Unexpected stop reason: {response.stop_reason}]"
                break

        else:
            # ── Safeguard: iteration cap reached ──────────────────────────
            final_text = (
                "[AGENT ABORTED] Maximum tool-call iterations reached. "
                "Escalating to human operator for manual resolution."
            )
            self.tools.escalate_to_human(
                reason  = f"Iteration cap reached on query: {user_query[:80]}",
                urgency = "medium",
            )
            if verbose:
                print(f"\nAGENT (aborted): {final_text}")

        # ── Log the completed decision ────────────────────────────────────
        self.memory.log_decision(
            query   = user_query,
            action  = "agent_response",
            outcome = final_text[:120],
        )

        return final_text


print("WarehouseAgent class defined.")

WarehouseAgent class defined.


<a id='task4'></a>
---
## Task 4 — Execution and Observed Behavior

We run three representative scenarios to observe the agent's decision logic, tool-use patterns, and safeguard activation. The agent is instantiated once and reused across scenarios so the decision log accumulates realistically.